# RQ2_5 — Final Interpretation, Robustness & Validation

**Run after:** `RQ2_1_JIRA_Extraction.ipynb`, `RQ2_2_Reassignments.ipynb`, `RQ2_3_Analysis.ipynb`, and `RQ2_4_EDA.ipynb`.

## Purpose

This notebook is the robustness and interpretation layer for RQ2. It does **not replace RQ2_3**. The original OLS model remains the primary analysis, while this notebook tests whether the main conclusions are stable under alternative outcome and reassignment specifications.

### RQ2 focus

> **How do issue characteristics, including reassignment activity, relate to issue resolution time, and did these relationships differ between the pre-AI and AI-era periods?**

### Robustness strategy

1. Validate the analytical sample and project × era composition.
2. Re-estimate the original raw-resolution-time OLS model.
3. Add `log1p(resolution_time_days)` OLS because resolution time is highly right-skewed.
4. Add a Gamma GLM with log link as a distribution-sensitive robustness model.
5. Test reassignment count as continuous, binary, and categorical.
6. Test priority × era, comments × era, and reassignment × era interactions.
7. Compare effect sizes and model fit.
8. Produce a final evidence matrix and report-ready interpretation.

> **Causal caution:** AI era is a temporal proxy. Reassignment is an observed issue-history characteristic. Neither establishes that AI or reassignment caused a change in resolution time.


In [11]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from pathlib import Path

BASE = Path(".")

JIRA_FILE = "apache_jira_raw.csv"
REASSIGN_FILE = "num_reassignments_real.csv"
TARGET = "resolution_time_days"

print("RQ2_5 FINAL loaded.")
print("JIRA source:", JIRA_FILE)
print("Reassignment source:", REASSIGN_FILE)


RQ2_5 FINAL loaded.
JIRA source: apache_jira_raw.csv
Reassignment source: num_reassignments_real.csv


In [12]:
# IMPORTANT:
# num_reassignments_real.csv contains only:
#   issue_id, num_reassignments
# Resolution time, priority, comments, project and dates come from
# apache_jira_raw.csv, so the two real outputs must be merged exactly
# as in RQ2_3.

jira = pd.read_csv(BASE / JIRA_FILE)
reassign = pd.read_csv(BASE / REASSIGN_FILE)

merged = jira.merge(reassign, on="issue_id", how="inner")

merged["resolution_date_parsed"] = pd.to_datetime(
    merged["resolution_date"], errors="coerce", utc=True
)
merged["created_parsed"] = pd.to_datetime(
    merged["created"], errors="coerce", utc=True
)

merged["era"] = (
    merged["resolution_date_parsed"] >= pd.Timestamp("2023-01-01", tz="UTC")
).map({True: "ai_era", False: "pre_ai"})

merged["resolution_time_days"] = (
    merged["resolution_date_parsed"] - merged["created_parsed"]
).dt.total_seconds() / 86400

merged["era_binary"] = (merged["era"] == "ai_era").astype(int)

# Match RQ2_3 cleaning logic.
analysis = merged.dropna(
    subset=["resolution_time_days", "priority", "num_comments", "num_reassignments"]
).copy()

analysis = analysis[analysis["resolution_time_days"] >= 0].copy()

# Preserve the source terminology: num_comments, not num_comments.
analysis["num_comments"] = pd.to_numeric(
    analysis["num_comments"], errors="coerce"
)
analysis["num_reassignments"] = pd.to_numeric(
    analysis["num_reassignments"], errors="coerce"
)

analysis = analysis.dropna(
    subset=["resolution_time_days", "num_comments", "num_reassignments"]
).copy()

print(f"Merged sample: {len(merged)} issues")
print(f"Usable analytical sample: {len(analysis)}")
print("\nEra counts:")
display(analysis["era"].value_counts())

print("\nProject × era counts:")
if "project_name" in analysis.columns:
    display(
        analysis.groupby(["project_name", "era"])
        .size()
        .unstack(fill_value=0)
    )

print("\nColumns available for RQ2_5:")
print(analysis.columns.tolist())


Merged sample: 3000 issues
Usable analytical sample: 2993

Era counts:


,count
era,
pre_ai,2452
ai_era,541



Project × era counts:


era,ai_era,pre_ai
project_name,,
CAMEL,472,1506
HADOOP,69,946



Columns available for RQ2_5:
['issue_id', 'project_name', 'created', 'resolution_date', 'priority', 'component', 'num_comments', 'num_reassignments', 'resolution_date_parsed', 'created_parsed', 'era', 'resolution_time_days', 'era_binary']


## 1. Sample and distribution validation

The original RQ2 analysis uses the reassignment-linked analytical dataset. This cell documents the final usable N and the extreme skew in resolution time.

A large difference between mean and median indicates that raw OLS should be complemented by a transformed/distribution-sensitive model.


In [13]:
dist = pd.DataFrame({
    "Statistic": ["N", "Mean", "Median", "Std Dev", "Minimum", "Maximum",
                  "90th percentile", "95th percentile", "99th percentile"],
    "Resolution time (days)": [
        len(analysis),
        analysis[TARGET].mean(),
        analysis[TARGET].median(),
        analysis[TARGET].std(),
        analysis[TARGET].min(),
        analysis[TARGET].max(),
        analysis[TARGET].quantile(.90),
        analysis[TARGET].quantile(.95),
        analysis[TARGET].quantile(.99)
    ]
})
display(dist.round(3))

print("\nReassignments: zero vs one or more")
analysis["reassignment_any"] = (analysis["num_reassignments"] >= 1).astype(int)
display(
    analysis["reassignment_any"]
    .value_counts()
    .rename(index={0:"0 reassignments",1:"≥1 reassignment"})
    .rename("N").to_frame()
)


,Statistic,Resolution time (days)
0,N,2993.000
1,Mean,72.192
2,Median,3.339
3,Std Dev,260.731
4,Minimum,0.000
5,Maximum,3511.024
6,90th percentile,147.758
7,95th percentile,346.648
8,99th percentile,1414.713



Reassignments: zero vs one or more


,N
reassignment_any,
≥1 reassignment,1603
0 reassignments,1390


## 2. Primary RQ2 model reproduction

This reproduces the conceptual structure of RQ2_3:

> resolution time ~ priority + comments + reassignments + AI era + priority×era + comments×era + reassignments×era

This raw-scale OLS model remains the **primary reference model**. The models below are robustness checks.


In [14]:
formula_primary = (
    f"{TARGET} ~ C(priority) + num_comments + num_reassignments + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + num_reassignments:era_binary"
)

ols_raw = smf.ols(formula_primary, data=analysis).fit(cov_type="HC3")

print(ols_raw.summary())

raw_params = pd.DataFrame({
    "Coefficient": ols_raw.params,
    "p-value": ols_raw.pvalues,
    "CI low": ols_raw.conf_int()[0],
    "CI high": ols_raw.conf_int()[1]
})

display(raw_params.round(4))


                             OLS Regression Results                             
Dep. Variable:     resolution_time_days   R-squared:                       0.079
Model:                              OLS   Adj. R-squared:                  0.075
Method:                   Least Squares   F-statistic:                     5.162
Date:                  Tue, 18 Aug 2026   Prob (F-statistic):           3.54e-09
Time:                          10:17:20   Log-Likelihood:                -20774.
No. Observations:                  2993   AIC:                         4.158e+04
Df Residuals:                      2979   BIC:                         4.166e+04
Df Model:                            13                                         
Covariance Type:                    HC3                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

,Coefficient,p-value,CI low,CI high
Intercept,-66.1831,0.0146,-119.2967,-13.0695
C(priority)[T.Critical],68.1021,0.0642,-4.0196,140.2238
C(priority)[T.Major],67.0954,0.0059,19.3654,114.8253
C(priority)[T.Minor],77.7383,0.0045,24.1587,131.3180
C(priority)[T.Trivial],71.7668,0.0604,-3.1426,146.6762
num_comments,4.6799,0.0000,2.8983,6.4615
num_reassignments,61.8458,0.0000,38.5405,85.1512
era_binary,594.7125,0.3844,-745.2213,1934.6463
C(priority)[T.Critical]:era_binary,-662.3277,0.3406,-2024.5595,699.9041
C(priority)[T.Major]:era_binary,-603.1489,0.3827,-1957.4490,751.1512


## 3. Log-transformed outcome robustness

Resolution time is strongly right-skewed. Therefore, the same predictor structure is estimated with:

`log1p(resolution_time_days)`

This does not replace the raw OLS result. It tests whether the direction and statistical evidence are stable when the long right tail is compressed.


In [15]:
analysis["log_resolution_time"] = np.log1p(analysis[TARGET].clip(lower=0))

formula_log = (
    "log_resolution_time ~ C(priority) + num_comments + num_reassignments + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + num_reassignments:era_binary"
)

ols_log = smf.ols(formula_log, data=analysis).fit(cov_type="HC3")

log_params = pd.DataFrame({
    "Coefficient": ols_log.params,
    "p-value": ols_log.pvalues,
    "CI low": ols_log.conf_int()[0],
    "CI high": ols_log.conf_int()[1]
})

print("Log-outcome OLS:")
display(log_params.round(4))

print("Raw OLS R²:", round(ols_raw.rsquared, 4))
print("Log OLS R²:", round(ols_log.rsquared, 4))


Log-outcome OLS:


,Coefficient,p-value,CI low,CI high
Intercept,0.6511,0.0003,0.3004,1.0018
C(priority)[T.Critical],0.6500,0.0380,0.0360,1.2639
C(priority)[T.Major],0.5387,0.0022,0.1938,0.8835
C(priority)[T.Minor],0.5192,0.0048,0.1587,0.8797
C(priority)[T.Trivial],0.1321,0.6237,-0.3956,0.6597
num_comments,0.0691,0.0000,0.0588,0.0793
num_reassignments,0.6666,0.0000,0.5596,0.7735
era_binary,0.8395,0.5615,-1.9945,3.6736
C(priority)[T.Critical]:era_binary,-0.7027,0.6521,-3.7573,2.3520
C(priority)[T.Major]:era_binary,-0.7273,0.6173,-3.5806,2.1259


Raw OLS R²: 0.0793
Log OLS R²: 0.2156


## 4. Gamma GLM robustness

A Gamma generalized linear model with a log link provides an additional distribution-sensitive check for a positive, right-skewed duration outcome.

This model is included as a robustness analysis. It is not treated as evidence of causality.


In [16]:
# Gamma requires strictly positive outcomes.
gamma_df = analysis[analysis[TARGET] > 0].copy()

gamma_model = smf.glm(
    formula=formula_primary,
    data=gamma_df,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

gamma_params = pd.DataFrame({
    "Coefficient": gamma_model.params,
    "p-value": gamma_model.pvalues,
    "CI low": gamma_model.conf_int()[0],
    "CI high": gamma_model.conf_int()[1]
})

print(gamma_model.summary())
print("\nGamma GLM coefficients:")
display(gamma_params.round(4))


                  Generalized Linear Model Regression Results                   
Dep. Variable:     resolution_time_days   No. Observations:                 2993
Model:                              GLM   Df Residuals:                     2979
Model Family:                     Gamma   Df Model:                           13
Link Function:                      Log   Scale:                          15.792
Method:                            IRLS   Log-Likelihood:                -12545.
Date:                  Tue, 18 Aug 2026   Deviance:                       17736.
Time:                          10:17:21   Pearson chi2:                 4.70e+04
No. Iterations:                      94   Pseudo R-squ. (CS):            0.03447
Covariance Type:              nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

,Coefficient,p-value,CI low,CI high
Intercept,2.3948,0.0000,1.5371,3.2525
C(priority)[T.Critical],0.9341,0.1936,-0.4741,2.3424
C(priority)[T.Major],0.9510,0.0273,0.1067,1.7954
C(priority)[T.Minor],0.9075,0.0431,0.0284,1.7867
C(priority)[T.Trivial],0.9251,0.1355,-0.2897,2.1400
num_comments,0.0452,0.0000,0.0291,0.0612
num_reassignments,0.6555,0.0000,0.4256,0.8854
era_binary,1.0479,0.6117,-2.9985,5.0943
C(priority)[T.Critical]:era_binary,-3.1552,0.3699,-10.0524,3.7421
C(priority)[T.Major]:era_binary,-1.2687,0.5374,-5.3005,2.7631


## 5. Reassignment specification robustness

The original analysis treats `num_reassignments` as a continuous count. Because almost half of the analytical issues have zero reassignments and the maximum is six, we test three specifications:

1. **Continuous count**
2. **Binary:** 0 vs ≥1
3. **Categorical:** 0, 1, 2+

If the substantive conclusion is stable across these specifications, confidence in the reassignment finding increases.


In [17]:
analysis["reassignment_cat"] = pd.cut(
    analysis["num_reassignments"],
    bins=[-0.1, 0.5, 1.5, np.inf],
    labels=["0", "1", "2+"]
)

# Binary reassignment model
binary_formula = (
    f"{TARGET} ~ C(priority) + num_comments + reassignment_any + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + reassignment_any:era_binary"
)
ols_binary = smf.ols(binary_formula, data=analysis).fit(cov_type="HC3")

# Categorical reassignment model
cat_formula = (
    f"{TARGET} ~ C(priority) + num_comments + C(reassignment_cat) + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary"
)
ols_cat = smf.ols(cat_formula, data=analysis).fit(cov_type="HC3")

reassignment_summary = pd.DataFrame({
    "Specification": [
        "Continuous count",
        "0 vs ≥1",
        "0 / 1 / 2+"
    ],
    "R²": [
        ols_raw.rsquared,
        ols_binary.rsquared,
        ols_cat.rsquared
    ],
    "Reassignment evidence": [
        f"coef={ols_raw.params.get('num_reassignments', np.nan):.4f}, p={ols_raw.pvalues.get('num_reassignments', np.nan):.4g}",
        f"coef={ols_binary.params.get('reassignment_any', np.nan):.4f}, p={ols_binary.pvalues.get('reassignment_any', np.nan):.4g}",
        "Joint/category comparison shown below"
    ]
})

display(reassignment_summary.round(4))
display(pd.DataFrame({
    "Coefficient": ols_cat.params,
    "p-value": ols_cat.pvalues,
    "CI low": ols_cat.conf_int()[0],
    "CI high": ols_cat.conf_int()[1]
}).loc[[x for x in ols_cat.params.index if "reassignment_cat" in x]].round(4))


,Specification,R²,Reassignment evidence
0,Continuous count,0.0793,"coef=61.8458, p=1.98e-07"
1,0 vs ≥1,0.0597,"coef=53.1911, p=5.433e-09"
2,0 / 1 / 2+,0.0852,Joint/category comparison shown below


,Coefficient,p-value,CI low,CI high
C(reassignment_cat)[T.1],30.0587,0.0002,14.3007,45.8167
C(reassignment_cat)[T.2+],233.8777,0.0000,137.5157,330.2397


## 6. AI-era moderation

The key RQ2 moderation question is whether the relationship between reassignment activity and resolution time changed in the AI era.

The primary term is:

`num_reassignments × era_binary`

The same logic is applied to priority and comments.

A non-significant reassignment × era interaction means that the evidence does **not** support a statistically detectable change in the reassignment–resolution-time association across the two periods.


In [18]:
interaction_terms = [
    "C(priority):era_binary",
    "num_comments:era_binary",
    "num_reassignments:era_binary"
]

interaction_results = pd.DataFrame({
    "Interaction": interaction_terms,
    "Coefficient": [ols_raw.params.get(x, np.nan) for x in interaction_terms],
    "p-value": [ols_raw.pvalues.get(x, np.nan) for x in interaction_terms],
    "CI low": [ols_raw.conf_int().loc[x, 0] if x in ols_raw.params else np.nan for x in interaction_terms],
    "CI high": [ols_raw.conf_int().loc[x, 1] if x in ols_raw.params else np.nan for x in interaction_terms]
})

interaction_results["Significant at .05"] = interaction_results["p-value"] < 0.05
display(interaction_results.round(4))


,Interaction,Coefficient,p-value,CI low,CI high,Significant at .05
0,C(priority):era_binary,NaN,NaN,NaN,NaN,False
1,num_comments:era_binary,-0.9983,0.6199,-4.9433,2.9467,False
2,num_reassignments:era_binary,26.4984,0.6273,-80.4767,133.4735,False


## 7. Effect-size and incremental explanatory value

The original RQ2 analysis reports the increase in R² after adding reassignment information. This section retains that logic and reports Cohen's f².

For the raw-scale OLS comparison:

- **Reduced model:** priority + comments + era + their era interactions
- **Extended model:** reduced model + reassignment + reassignment × era

Cohen's f² is:

`(R²_full - R²_reduced) / (1 - R²_full)`


In [19]:
reduced_formula = (
    f"{TARGET} ~ C(priority) + num_comments + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary"
)

reduced_model = smf.ols(reduced_formula, data=analysis).fit(cov_type="HC3")

r2_reduced = reduced_model.rsquared
r2_full = ols_raw.rsquared
delta_r2 = r2_full - r2_reduced
f2 = delta_r2 / (1 - r2_full)

effect_summary = pd.DataFrame([{
    "Reduced R²": r2_reduced,
    "Full R²": r2_full,
    "Incremental R²": delta_r2,
    "Cohen f²": f2,
    "Reassignment coefficient": ols_raw.params.get("num_reassignments", np.nan),
    "Reassignment p-value": ols_raw.pvalues.get("num_reassignments", np.nan),
    "Reassignment × Era p-value": ols_raw.pvalues.get("num_reassignments:era_binary", np.nan)
}])

display(effect_summary.round(4))


,Reduced R²,Full R²,Incremental R²,Cohen f²,Reassignment coefficient,Reassignment p-value,Reassignment × Era p-value
0,0.0513,0.0793,0.028,0.0304,61.8458,0.0,0.6273


## 8. Final evidence matrix

The evidence is intentionally separated into:

- primary raw-scale inference
- skew-robust outcome models
- reassignment specification robustness
- AI-era moderation
- incremental explanatory value

The final conclusion should distinguish the **main reassignment association** from the **AI-era moderation question**.


In [21]:
# ============================================================
# 8. FINAL EVIDENCE MATRIX — RQ2
# ============================================================

# Extract the key estimates and p-values safely
reassignment_coef = ols_raw.params.get(
    "num_reassignments", np.nan
)

reassignment_p = ols_raw.pvalues.get(
    "num_reassignments", np.nan
)

reassignment_era_coef = ols_raw.params.get(
    "num_reassignments:era_binary", np.nan
)

reassignment_era_p = ols_raw.pvalues.get(
    "num_reassignments:era_binary", np.nan
)

log_reassignment_coef = ols_log.params.get(
    "num_reassignments", np.nan
)

log_reassignment_p = ols_log.pvalues.get(
    "num_reassignments", np.nan
)

gamma_reassignment_coef = gamma_model.params.get(
    "num_reassignments", np.nan
)

gamma_reassignment_p = gamma_model.pvalues.get(
    "num_reassignments", np.nan
)


# ------------------------------------------------------------
# Build the evidence matrix
# ------------------------------------------------------------

evidence = pd.DataFrame([

    # --------------------------------------------------------
    # 1. Primary reassignment main effect
    # --------------------------------------------------------
    {
        "Evidence": "HC3-robust OLS: reassignment main effect",

        "Estimate": reassignment_coef,

        "p-value": reassignment_p,

        "Supported?": reassignment_p < 0.05,

        "Interpretation": (
            "Reassignment is significantly associated with "
            "longer resolution time."
            if reassignment_p < 0.05
            else
            "No statistically significant association detected."
        )
    },


    # --------------------------------------------------------
    # 2. Reassignment × AI-era interaction
    # --------------------------------------------------------
    {
        "Evidence": "HC3-robust OLS: reassignment × AI era",

        "Estimate": reassignment_era_coef,

        "p-value": reassignment_era_p,

        "Supported?": reassignment_era_p < 0.05,

        "Interpretation": (
            "Statistically significant evidence that the "
            "reassignment–resolution-time relationship changed "
            "in the AI era."
            if reassignment_era_p < 0.05
            else
            "No statistically significant evidence that the "
            "reassignment–resolution-time relationship changed "
            "in the AI era."
        )
    },


    # --------------------------------------------------------
    # 3. Log-transformed outcome robustness
    # --------------------------------------------------------
    {
        "Evidence": "Log-outcome OLS: reassignment",

        "Estimate": log_reassignment_coef,

        "p-value": log_reassignment_p,

        "Supported?": log_reassignment_p < 0.05,

        "Interpretation": (
            "The reassignment association remains statistically "
            "significant after log-transforming resolution time."
            if log_reassignment_p < 0.05
            else
            "The reassignment association is not statistically "
            "significant after log transformation."
        )
    },


    # --------------------------------------------------------
    # 4. Gamma GLM robustness
    # --------------------------------------------------------
    {
        "Evidence": "Gamma GLM: reassignment",

        "Estimate": gamma_reassignment_coef,

        "p-value": gamma_reassignment_p,

        "Supported?": gamma_reassignment_p < 0.05,

        "Interpretation": (
            "The reassignment association remains statistically "
            "significant under a Gamma GLM."
            if gamma_reassignment_p < 0.05
            else
            "The reassignment association is not statistically "
            "significant under a Gamma GLM."
        )
    },


    # --------------------------------------------------------
    # 5. Incremental explanatory value
    # --------------------------------------------------------
    {
        "Evidence": "Incremental R² from reassignment",

        "Estimate": delta_r2,

        "p-value": np.nan,

        "Supported?": delta_r2 > 0,

        "Interpretation": (
            f"Adding reassignment increases explained variance "
            f"by {delta_r2:.4f}."
            if delta_r2 > 0
            else
            "Adding reassignment does not increase explained variance."
        )
    }

])


# ------------------------------------------------------------
# Display the final evidence matrix
# ------------------------------------------------------------

display(
    evidence.round(5)
)

,Evidence,Estimate,p-value,Supported?,Interpretation
0,HC3-robust OLS: reassignment main effect,61.84582,0.00000,True,Reassignment is significantly associated with ...
1,HC3-robust OLS: reassignment × AI era,26.49840,0.62733,False,No statistically significant evidence that the...
2,Log-outcome OLS: reassignment,0.66658,0.00000,True,The reassignment association remains statistic...
3,Gamma GLM: reassignment,0.65549,0.00000,True,The reassignment association remains statistic...
4,Incremental R² from reassignment,0.02799,NaN,True,Adding reassignment increases explained varian...


## 9. Report-ready interpretation

### Recommended interpretation logic

Use this hierarchy when writing RQ2:

1. **Primary association:** RQ2_3 raw-scale OLS model.
2. **Outcome robustness:** log-transformed OLS and Gamma GLM because resolution time is strongly right-skewed.
3. **Reassignment robustness:** continuous, binary, and categorical reassignment specifications.
4. **AI-era moderation:** reassignment × era, priority × era, and comments × era.
5. **Practical contribution:** incremental R² and Cohen's f².

### Recommended wording

> **Issue reassignment is positively associated with longer resolution time. In the primary OLS model, each additional recorded reassignment is associated with an estimated increase in resolution time, and adding reassignment information provides incremental explanatory value beyond priority, comments, and AI-era status. This relationship should be interpreted as an association rather than a causal effect. Robustness analyses using a log-transformed resolution-time outcome and a Gamma GLM are used to assess whether the finding is sensitive to the substantial right skew in resolution time.**

> **The reassignment × AI-era interaction is evaluated separately from the reassignment main effect. If the interaction remains non-significant across the robustness models, the appropriate conclusion is that there is no statistically detectable evidence that the reassignment–resolution-time relationship changed between the pre-AI and AI-era periods.**

### Do not write

- "Reassignments cause delays."
- "AI caused faster/slower resolution."
- "AI reduced the impact of reassignment."
- "The model proves that AI changed issue resolution."

### Preferred terminology

Use:

- "associated with"
- "estimated increase/decrease"
- "resolution time"
- "reassignment activity"
- "AI-era temporal proxy"
- "incremental explanatory value"
- "robustness analysis"
- "statistically detectable moderation"
- "does not establish causality".


## 10. Methodological decision rule

### RQ2 main-effect decision

If the reassignment coefficient remains positive and statistically significant across the raw OLS, log-outcome OLS, and Gamma GLM, the evidence for a robust association between reassignment activity and longer resolution time is strengthened.

### RQ2 AI-era decision

If `num_reassignments × era_binary` remains non-significant, report:

> **There is no statistically detectable evidence that the reassignment–resolution-time relationship changed between the pre-AI and AI-era periods.**

This does **not** prove that the relationship is identical across eras; it means that the available evidence does not detect a statistically significant difference.

### Sampling terminology

If the RQ2_2 notebook uses `DataFrame.sample()` without explicit strata, describe the 3,000-issue selection as a **random sample**, not a stratified sample. Only use "stratified sample" if the sampling code explicitly enforces project/era strata.

### Causality rule

Neither reassignment nor AI era should be described as a causal mechanism from these observational analyses alone.
